# Lab 01 — Checagens de qualidade como queries (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite).

Todo teste de dados é **uma query que busca violações** (0 = passa). Vamos ver as 4 clássicas.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
con.execute('CREATE TABLE pedidos(id INT, status VARCHAR, valor INT)')
con.executemany('INSERT INTO pedidos VALUES (?,?,?)', [
    (1,'pago',100),(2,'cancelado',50),(2,'pago',80),(3,'X',-10),(4,'pago',None)])
con.execute('SELECT * FROM pedidos').df()

## not_null (completude): linhas com valor nulo

In [ ]:
con.execute('SELECT COUNT(*) AS nulos FROM pedidos WHERE valor IS NULL').df()

## unique (unicidade): ids duplicados

In [ ]:
con.execute('SELECT id, COUNT(*) c FROM pedidos GROUP BY id HAVING COUNT(*)>1').df()

## accepted_values (validade): status fora do domínio

In [ ]:
con.execute("SELECT DISTINCT status FROM pedidos WHERE status NOT IN ('pago','cancelado','enviado')").df()

## Sua vez: quantos pedidos têm valor NEGATIVO (regra de negócio)? Verifique.

In [ ]:
resposta = con.execute('SELECT COUNT(*) FROM pedidos WHERE valor < 0').fetchone()[0]
resposta

In [ ]:
def verificar(v):
    try:
        assert v == 1, 'Só o pedido 3 (valor -10).'
        print('✅ Correto! Cada regra vira uma query que busca violações (0 = passa).')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)